# 12 · Distributed ML with Spark DataFrames

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** databricks. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Assemble features, fit a Spark ML pipeline, and score without collecting all rows.


## Step 1 — Create a Spark dataset

Use DataFrame APIs compatible with Spark Connect. Spark ML support depends on the attached compute/environment version; use a recent supported environment. Notebook 06 is the CPU alternative.


In [ ]:
import hashlib
from pyspark.sql import functions as F
user = spark.sql('SELECT current_user()').first()[0]
catalog = spark.sql('SELECT current_catalog()').first()[0]
schema = 'ai_lab_' + hashlib.sha256(user.encode()).hexdigest()[:10]
# Uses the current catalog; change it to a catalog where you can CREATE SCHEMA.
namespace = f'`{catalog}`.`{schema}`'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {namespace}')
def table(name):
    assert name.replace('_', '').isalnum()
    return f'{namespace}.`{name}`'
print('Course namespace:', namespace)

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
df=spark.range(1200).withColumn('x1',F.rand(42)).withColumn('x2',F.rand(7))
df=df.withColumn('label',((F.col('x1') + 0.4*F.col('x2')) > 0.7).cast('double'))
train,test=df.randomSplit([0.8,0.2],42)
assert train.count() > 0 and test.count() > 0


**Check your result:** Training and test partitions are nonempty.


## Step 2 — Fit a distributed pipeline

VectorAssembler creates a features vector. Scaling is fitted only on training data.


In [ ]:
pipeline=Pipeline(stages=[
    VectorAssembler(inputCols=['x1','x2'],outputCol='raw_features'),
    StandardScaler(inputCol='raw_features',outputCol='features',withMean=False),
    LogisticRegression(featuresCol='features',labelCol='label',maxIter=30)])
model=pipeline.fit(train)
predictions=model.transform(test)
display(predictions.select('id','label','probability','prediction').limit(10))


**Check your result:** One probability vector and prediction exist per scored row.


## Step 3 — Evaluate in Spark

Only a scalar metric returns to the driver. Avoid collecting a large feature matrix.


In [ ]:
auc=BinaryClassificationEvaluator(labelCol='label',metricName='areaUnderROC').evaluate(predictions)
assert auc > 0.9
print('Spark test AUC:',auc)


**Check your result:** The simple separable dataset produces strong AUC.


## Practice

Add a third irrelevant feature, rerun, and compare AUC and training time.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Assemble all three input columns. Compare on the same split; a useless feature does not necessarily cause a large metric change.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
